# Pipeline SDC — Analyse de métadonnées par LLM

Ce notebook présente le pipeline `metadata-analysis-llm-for-sdc` : il transforme un
classeur de métadonnées en un **tableau plat normalisé** pour rtauargus.

Run pip install -r requirements-notebook.txt

## Vue d'ensemble

| Module | Rôle |
|---|---|
| `read_input.py` | Classeur (.ods/.xlsx/.csv) → Markdown — **déterministe** |
| `llm_client.py` | Appel au modèle Qwen (API compatible OpenAI) |
| `pipeline.py` | Orchestrateur : sérialisation → Phase 1 → Phase 2 |
| `extract_json.py` | Valide le JSON du modèle contre le schéma |
| `json_to_table.py` | JSON validé → `.csv` (humain) |
| `run_full_pipeline_on_minio.py` | Pipeline complet MinIO → MinIO (script terminal) |

**Flux** :
```
classeur (MinIO)
  → serialize          déterministe  → Markdown
  → Phase 1 (LLM)     probabiliste  → questions au producteur  ← vous répondez ici
  → Phase 2 (LLM)     probabiliste  → JSON validé
  → write_csv          déterministe  → CSV (MinIO)
```


## 1. Mise en place

> **Prérequis Onyxia :** service VSCode-python lancé avec `CLE_API_OPENWEBUI` déclarée en variable d'environnement, et `pip install -r requirements.txt` exécuté dans le terminal. EDGE FONCTIONNE MIEUX QUE FIREFOX.

In [ ]:
import os, sys, tempfile
from pathlib import Path
import s3fs
from IPython.display import Markdown, display

# Détection de la racine du repo (fonctionne que le notebook soit lancé
# depuis la racine ou depuis notebooks/)
for _p in [Path.cwd(), Path.cwd().parent]:
    if (_p / "core").is_dir():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Impossible de localiser 'core/'. Lancez Jupyter depuis la racine du repo.")

sys.path.insert(0, str(REPO_ROOT))
from core import pipeline, extract_json, json_to_table

# Vérification de l'environnement
api_key = os.environ.get("CLE_API_OPENWEBUI") or os.environ.get("OPENAI_API_KEY")
assert api_key, "Clé API introuvable — déclarez CLE_API_OPENWEBUI dans les variables d'environnement Onyxia."
print(f"✓ Clé API       : {'*' * (len(api_key))}")
print(f"✓ Modèle        : {os.environ.get('LLM_MODEL', 'qwen3-6-35b-moe')}")

fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://" + os.environ["AWS_S3_ENDPOINT"]}
)
print("✓ MinIO         : connexion initialisée")

## 2. Sérialisation — ce que le modèle reçoit

`read_input.serialize` convertit le classeur en Markdown pur.
Cette étape est **100 % déterministe** : même fichier → même Markdown.


In [ ]:
INPUT_S3  = "[Chemin_fichier]/[nom_fichier].ods"
OUTPUT_S3 = "[Chemin_fichier_output]/[nom_fichier_output].csv"

# Téléchargement depuis MinIO vers /tmp
tmp_input = Path(f"/tmp/input{Path(INPUT_S3).suffix}")
with fs.open(INPUT_S3, "rb") as f_in:
    tmp_input.write_bytes(f_in.read())
print(f"✓ Téléchargé : {INPUT_S3}\n")

# Sérialisation
md_input = pipeline.serialize(tmp_input)
print(f"  {len(md_input)} caractères — aperçu :")
display(Markdown(md_input))

## 3. Phase 1 — le modèle analyse et pose ses questions

`pipeline.start()` envoie le Markdown au modèle avec les instructions.

Le modèle produit :
- des **notes de travail** (avant le séparateur `---`)
- puis soit une **liste de questions** (si des ambiguïtés subsistent), soit `Aucune question.`
  suivi du JSON directement

> ⏳ Cette cellule appelle le LLM — comptez environ 1 à 3 minutes.


In [ ]:
print("Phase 1 — envoi au modèle...\n")
r = pipeline.start(md_input)

if r.auto_continued:
    print("Le modèle n'a posé aucune question — JSON produit directement.")
    print("Passez à la cellule Phase 2 (la cellule 'Réponses' sera ignorée).")
else:
    print("─" * 70)
    print(r.questions)
    print("─" * 70)
    print("\n→ Remplissez la variable `answers` dans la cellule ci-dessous, puis exécutez-la.")

## 4. Réponses du producteur

Remplissez la variable `answers` avec vos réponses aux questions ci-dessus
(une réponse par question, dans l'ordre), puis exécutez la cellule.


In [ ]:
# Remplissez vos réponses ici avant d'exécuter cette cellule.
answers = """

"""

print("Réponses enregistrées — exécutez la cellule Phase 2.")

## 5. Phase 2 — JSON, validation et CSV final

`pipeline.answer()` envoie vos réponses au modèle qui produit le JSON.
Le JSON est ensuite **validé contre le schéma** (étape déterministe), puis
converti en tableau plat et écrit sur MinIO.

> ⏳ Cette cellule appelle à nouveau le LLM — comptez environ 1 à 2 minutes.


In [ ]:
import pandas as pd

if r.auto_continued:
    # Phase 1 a déjà produit le JSON — pas besoin d'un second appel
    records = r.records
    print("Résultats issus de la Phase 1 (aucune question posée).\n")
else:
    print("Phase 2 — envoi des réponses au modèle...\n")
    records = pipeline.answer(r.history, answers)
    print(f"✓ {len(records)} enregistrement(s) validés contre le schéma.\n")

# Écriture du CSV sur MinIO
tmp_csv = Path("/tmp/output.csv")
cols, rows = json_to_table.write_csv(records, tmp_csv)

with open(tmp_csv, "r", encoding="utf-8-sig") as f:
    csv_content = f.read()
with fs.open(OUTPUT_S3, "w", encoding="utf-8") as f_out:
    f_out.write(csv_content)

print(f"✓ CSV écrit sur MinIO : {OUTPUT_S3}")
print(f"  {len(rows)} lignes × {len(cols)} colonnes\n")

# Affichage du résultat
df = pd.read_csv(tmp_csv, encoding="utf-8-sig", keep_default_na=False)
display(df)

# 2B: Un exemple plus simple
### Insérer un exemple sans ambiguités.

In [ ]:
# Charger le fichier

INPUT_S3  = "[Chemin_fichier]/[nom_fichier].ods"
OUTPUT_S3 = "[Chemin_fichier_output]/[nom_fichier_output].csv"

# Téléchargement depuis MinIO vers /tmp
tmp_input = Path(f"/tmp/input{Path(INPUT_S3).suffix}")
with fs.open(INPUT_S3, "rb") as f_in:
    tmp_input.write_bytes(f_in.read())
print(f"✓ Téléchargé : {INPUT_S3}\n")

# Sérialisation
md_input = pipeline.serialize(tmp_input)
print(f"  {len(md_input)} caractères — aperçu :")
display(Markdown(md_input))

In [ ]:
# Appel 1 au LLM 

print("Phase 1 — envoi au modèle...\n")
r = pipeline.start(md_input)

if r.auto_continued:
    print("Le modèle n'a posé aucune question — JSON produit directement.")
    print("Passez à la cellule Phase 2 (la cellule 'Réponses' sera ignorée).")
else:
    print("─" * 70)
    print(r.questions)
    print("─" * 70)
    print("\n→ Remplissez la variable `answers` dans la cellule ci-dessous, puis exécutez-la.")

In [ ]:
# reponse aux questions: 

# Remplissez vos réponses ici avant d'exécuter cette cellule.
answers = """

"""

print("Réponses enregistrées — exécutez la cellule Phase 2.")


In [ ]:
# output Json et formattage csv

import pandas as pd

if r.auto_continued:
    # Phase 1 a déjà produit le JSON — pas besoin d'un second appel
    records = r.records
    print("Résultats issus de la Phase 1 (aucune question posée).\n")
else:
    print("Phase 2 — envoi des réponses au modèle...\n")
    records = pipeline.answer(r.history, answers)
    print(f"✓ {len(records)} enregistrement(s) validés contre le schéma.\n")

# Écriture du CSV sur MinIO
tmp_csv = Path("/tmp/output.csv")
cols, rows = json_to_table.write_csv(records, tmp_csv)

with open(tmp_csv, "r", encoding="utf-8-sig") as f:
    csv_content = f.read()
with fs.open(OUTPUT_S3, "w", encoding="utf-8") as f_out:
    f_out.write(csv_content)

print(f"✓ CSV écrit sur MinIO : {OUTPUT_S3}")
print(f"  {len(rows)} lignes × {len(cols)} colonnes\n")

# Affichage du résultat
df = pd.read_csv(tmp_csv, encoding="utf-8-sig", keep_default_na=False)
display(df)